In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ir_rpp.preprocessing import load_dfs
from ir_rpp.statistical_tests import run_ttests, run_tukeys_hsd_test


%load_ext autoreload
%autoreload 2

In [12]:
summary, df_preference, df_metric = load_dfs("beerAdvocate", binary_relevance=3.5)

Reading run files: 100%|██████████| 21/21 [01:15<00:00,  3.59s/it]


Iterating over qids:   0%|          | 0/17564 [00:00<?, ?it/s]

In [14]:
df_ttest, df_summary = run_ttests(df_preference)
df_summary

metric,rpp,invrpp,dcgrpp,ap,ndcg,rr
significant,95.24,95.24,95.24,93.33,97.14,93.33


In [15]:
df_tukey, summary = run_tukeys_hsd_test(
    df_preference,
    metrics=["rpp", "invrpp", "dcgrpp", "ap", "ndcg", "rr"],
    n_permutations=2000
)

Comparing Pairs: 100%|██████████| 210/210 [00:00<00:00, 76498.51it/s]


In [16]:
summary

metric,rpp,invrpp,dcgrpp,ap,ndcg,rr
significant,93.81,94.29,93.33,92.38,94.29,91.9


### DL 2020 Passage

In [37]:
summary, df_preference, df_metric = load_dfs("dl20-passage", binary_relevance=2)

Reading run files: 100%|██████████| 59/59 [00:10<00:00,  5.51it/s]


Iterating over qids:   0%|          | 0/54 [00:00<?, ?it/s]

In [39]:
df_tukey, summary = run_tukeys_hsd_test(
    df_preference,
    metrics=["rpp", "invrpp", "dcgrpp", "ap", "ndcg", "rr"],
    n_permutations=2000
)

Comparing Pairs: 100%|██████████| 1711/1711 [00:00<00:00, 190069.50it/s]


In [40]:
summary

metric,rpp,invrpp,dcgrpp,ap,ndcg,rr
significant,12.22,13.21,13.33,7.83,7.19,0.06


## Load Data

In [ ]:
# Load data
rows = []
with open("../../data/beerAdvocate/all_results.jsonl", "r") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

metrics = ["rpp", "invrpp", "dcgrpp", "ap", "rbp", "rr", "ndcg", "rp"]

# Filter for preference rows only
# each row contains System A vs System B deltas for every query
df_pref = df[df['type'] == 'preference'].copy()

# Group by the pairs
pairs = df_pref.groupby(['runi', 'runj'])

results_table = []

for (run_i, run_j), group in pairs:
    pair_results = {"pair": f"{run_i}:{run_j}"}

    for m in metrics:
        # Distribution of differences (eg. SystemA_NDCG - SystemB_NDCG)
        scores = group[m].dropna()

        t_stat, p_val = ttest_1samp(scores, 0)
        
        pair_results[f"{m}_p"] = p_val
        pair_results[f"{m}_mean"] = scores.mean()
        
    results_table.append(pair_results)

df_stats = pd.DataFrame(results_table)

## t-test + Bonferroni

The table is based on pairwise differences.

In [17]:
df_hard = df_stats[abs(df_stats['ndcg_mean']) < 0.01]

alpha = 0.05
num_pairs = len(df_hard)
bonferroni_threshold = alpha / num_pairs # keep the overall error rate at 0.05 across all pairs
extreme_threshold = 1e-10

print(f"Percentage of significant differences (t-test, Bonferroni @ {bonferroni_threshold:.6e}):")
for m in metrics:
    sig_count = (df_hard[f"{m}_p"] < bonferroni_threshold).sum()
    percentage = (sig_count / num_pairs) * 100
    print(f"{m:15}: {percentage:.2f}%")

Percentage of significant differences (t-test, Bonferroni @ 2.000000e-03):
rpp            : 76.00%
invrpp         : 84.00%
dcgrpp         : 76.00%
ap             : 68.00%
rbp            : 60.00%
rr             : 56.00%
ndcg           : 84.00%
rp             : 76.00%


## Tukey’s HSD

In [22]:
import json 

# Retrieve the absolute metric scores
metric_rows = []
with open("../../data/beerAdvocate/all_results.jsonl", "r") as f:
    for line in f:
        data = json.loads(line)
        if data.get("type") == "metric":
            metric_rows.append(data)

df_absolute = pd.DataFrame(metric_rows)
df_absolute.dtypes

KeyboardInterrupt: 

In [ ]:

print("=== Table 3: Tukey's HSD Sensitivity ===")
for m in ["ap", "ndcg", "rr", "rp"]:
    if m in df_absolute.columns:
        df_absolute[m] = pd.to_numeric(df_absolute[m], errors='coerce')
            
        res = pairwise_tukeyhsd(endog=df_absolute[m], groups=df_absolute['run'], alpha=0.05)
            
        sig_count = res.reject.sum()
        total_pairs = len(res.reject)
        percentage = (sig_count / total_pairs) * 100
        print(f"{m:10}: {percentage:.2f}% of pairs are significantly different")

# For RPP and variants (using the preference p-values in df_stats) use p < 0.05 without Bonferroni to match Tukey's sensitivity level
num_pairs = len(df_stats)
for m in ["rpp", "invrpp", "dcgrpp"]:
    sig_count = (df_stats[f"{m}_p"] < 0.05).sum()
    percentage = (sig_count / num_pairs) * 100
    print(f"{m:10}: {percentage:.2f}% of pairs are significantly different")